In [16]:
#Import libraries and load the dataset

import pandas as pd

df = pd.read_csv("teaching_prices_clean.csv")
df.head()

,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


In [17]:
#Inspect the dataset
print(df.shape)
print(df.columns)
df.info()

(96, 6)
Index(['date', 'item', 'city', 'price', 'unit', 'data_label'], dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        96 non-null     str    
 1   item        96 non-null     str    
 2   city        96 non-null     str    
 3   price       96 non-null     float64
 4   unit        96 non-null     str    
 5   data_label  96 non-null     str    
dtypes: float64(1), str(5)
memory usage: 14.4 KB


In [18]:
# Missing Values

print(df.isnull().sum())

date          0
item          0
city          0
price         0
unit          0
data_label    0
dtype: int64


In [19]:
# Research Question 1 — Did recorded prices increase between the first and last dates?

df['date'] = pd.to_datetime(df['date'])

earliest = df.sort_values('date').groupby(
    ['item', 'city']
).first()

latest = df.sort_values('date').groupby(
    ['item', 'city']
).last()

comparison = pd.DataFrame({
    'Earliest Price': earliest['price'],
    'Latest Price': latest['price']
})

comparison['Increase'] = (
    comparison['Latest Price'] - comparison['Earliest Price']
)

print(comparison)

                    Earliest Price  Latest Price  Increase
item        city                                          
Oil B       City 1           298.0         344.0      46.0
            City 2           304.0         350.0      46.0
Pulse C     City 1           200.0         207.5       7.5
            City 2           206.0         213.5       7.5
Staple A    City 1           101.0         114.5      13.5
            City 2           107.0         120.5      13.5
Sweetener D City 1           149.0         178.5      29.5
            City 2           155.0         184.5      29.5


In [20]:
# Research Question 2 — Were prices different between the two cities?

city_comparison = df.pivot_table(
    index=['date', 'item'],
    columns='city',
    values='price'
)

city_comparison['Difference'] = (
    city_comparison['City 2'] - city_comparison['City 1']
)

print(city_comparison)

city                    City 1  City 2  Difference
date       item                                   
2026-01-01 Oil B         298.0   304.0         6.0
           Pulse C       200.0   206.0         6.0
           Staple A      101.0   107.0         6.0
           Sweetener D   149.0   155.0         6.0
2026-01-08 Oil B         304.0   310.0         6.0
           Pulse C       202.5   208.5         6.0
           Staple A       99.5   105.5         6.0
           Sweetener D   153.5   159.5         6.0
2026-01-15 Oil B         310.0   316.0         6.0
           Pulse C       200.0   206.0         6.0
           Staple A      103.0   109.0         6.0
           Sweetener D   153.0   159.0         6.0
2026-01-22 Oil B         311.0   317.0         6.0
           Pulse C       202.5   208.5         6.0
           Staple A      106.5   112.5         6.0
           Sweetener D   157.5   163.5         6.0
2026-01-29 Oil B         317.0   323.0         6.0
           Pulse C       200.0 

In [21]:
# CELL 6: FIVE ALTERNATIVE QUESTIONS

# 1. Which item had the greatest price fluctuation?
fluctuation = df.groupby('item')['price'].max() - df.groupby('item')['price'].min()

print("1. Greatest price fluctuation:")
print(fluctuation.sort_values(ascending=False))


# 2. Which item had the most stable price?
print("\n2. Most stable item:")
print(fluctuation.sort_values())


# 3. Did any item experience a price decrease?
df_sorted = df.sort_values(['item', 'city', 'date'])

df_sorted['price_change'] = df_sorted.groupby(
    ['item', 'city']
)['price'].diff()

decreases = df_sorted[df_sorted['price_change'] < 0]

print("\n3. Price decreases:")
print(decreases[['date', 'item', 'city', 'price', 'price_change']])


# 4. What was the average price difference between cities for each item?
city_prices = df.pivot_table(
    index=['date', 'item'],
    columns='city',
    values='price'
)

city_prices['difference'] = (
    city_prices['City 2'] - city_prices['City 1']
)

print("\n4. Average city price difference:")
print(city_prices.groupby('item')['difference'].mean())


# 5. Which item had the highest average recorded price?
print("\n5. Average price of each item:")
print(df.groupby('item')['price'].mean().sort_values(ascending=False))

1. Greatest price fluctuation:
item
Oil B          52.0
Sweetener D    35.5
Staple A       22.5
Pulse C        13.5
Name: price, dtype: float64

2. Most stable item:
item
Pulse C        13.5
Staple A       22.5
Sweetener D    35.5
Oil B          52.0
Name: price, dtype: float64

3. Price decreases:
         date         item    city  price  price_change
20 2026-01-15      Pulse C  City 1  200.0          -2.5
36 2026-01-29      Pulse C  City 1  200.0          -2.5
60 2026-02-19      Pulse C  City 1  202.5          -2.5
76 2026-03-05      Pulse C  City 1  202.5          -2.5
21 2026-01-15      Pulse C  City 2  206.0          -2.5
37 2026-01-29      Pulse C  City 2  206.0          -2.5
61 2026-02-19      Pulse C  City 2  208.5          -2.5
77 2026-03-05      Pulse C  City 2  208.5          -2.5
8  2026-01-08     Staple A  City 1   99.5          -1.5
32 2026-01-29     Staple A  City 1  105.0          -1.5
48 2026-02-12     Staple A  City 1  107.0          -1.5
72 2026-03-05     Staple A  